# OpenMolcas SA-CASSCF-based NBRA-Workflow for Nonadiabatic Dynamics
In this tutorial, we demonstrate higher-level functions to streamline OpenMolcas calculations of the properties needed for NA-MD (nonadiabatic molecular dynamics) simulations. We will use these functions to define a workflow for NBRA (Neglect of back reaction approximation) calculations with OpenMolcas at the SA-CASSCF (State-Averaged Complete Active Space Self-Consistent Field) level of electronic structure.

The SA-CASSCF formalism provides:

Multiconfigurational description of excited states (essential for charge-transfer and multireference scenarios) State averaging across multiple roots for balanced treatment of nonadiabatic couplings Proper derivative couplings from the CI wavefunction structure Improved accuracy over simpler single-reference methods (TD-DFT, TD-DFTB) While the resulting functions are prototypes for general nonadiabatic calculations, at this point we will focus their use for NBRA-specific workflows involving:

Multiple excited-state trajectories Time-overlap matrix construction from CI vectors Derivative coupling extraction from finite-difference overlaps Fewest-switches surface hopping (FSSH) and related algorithms.

Output written in results_molcas:

    1. Adiabatic Hamiltoian matrix

    2. Adiabatic vibronic Hamiltonian matrix

    3. Atomic orbital overlap matrix

    4. Time overlap matrix

It also generates molden files written in wd_molcas_itraj0 for each time steps to visualize the NTOs.

In [3]:
"""
molcas_water_adi.py
-------------------
Nonadiabatic dynamics driver: trajectory with
OpenMolcas SA-CASSCF + full MRCI CI-state overlaps via Libra.
"""
import os
import sys
import copy
import warnings
import numpy as np
from types import SimpleNamespace
from liblibra_core import Py2Cpp_int, CMATRIX  
from libra_py import units
from libra_py.packages.cp2k import methods as cp2k

# working directory
PROJECT_DIR = "/user/someshch/project_molcas/LIBRA_MOLCAS"

_methods_path = os.path.join(PROJECT_DIR, "methods.py")
if not os.path.isfile(_methods_path):
    raise FileNotFoundError(
        f"Cannot find methods.py at:\n"
        f"  {_methods_path}\n"
        f"Make sure methods.py has been saved to PROJECT_DIR.\n"
        f"Current working directory: {os.getcwd()}"
    )

for _mod in list(sys.modules.keys()):
    if _mod == "methods" or _mod.startswith("methods."):
        del sys.modules[_mod]

if PROJECT_DIR in sys.path:
    sys.path.remove(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from methods import (
    run_molcas,
    read_molcas_orbital_info,
    read_ao_overlap,
    molcas_compute_adi,
    _infer_nstates_from_ciroot,
)

print(f"[import] Loaded methods.py from: {_methods_path}")

# ── MAIN ───────────────────────────────────────────────────────────────────

def main():
    os.chdir(PROJECT_DIR)
    # aligned nuclear steps
    labels, _ = cp2k.read_trajectory_xyz_file("water-md-aligned.xyz", 0)
    natoms = len(labels)
    print(f"Atoms  : {labels}")
    print(f"Natoms : {natoms}")

    molcas_run_params = {
        "basis"      : "ANO-RCC-VDZP",
        "charge"     : 0,
        "spin"       : 1,
        "title"      : "water-md",
        "nactel"     : "6 0 0",
        "inactive"   : 2,
        "ras2"       : 6,
        "ciroot"     : "4 4 1",
        "nac_states" : None,
        "group"      : "NoSym",
        "prwf"       : "1.0d-8",
        "thre"       : "1.0e-10",
        
    }

    params = {
        "atom_labels"                : labels,
        "timestep"                   : 0,
        "dt"                         : 0.5 * units.fs2au,
        "exe"                        : "/projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas",
        "molcas_run_params"          : molcas_run_params,
        "working_directory_prefix"   : "wd_molcas",
        "molcas_input_prefix"        : "input_",
        "molcas_output_prefix"       : "output_",
        "nstates"                    : 4,
        "ci_coeff_thresh"            : 1e-6,
        "verbose"                    : True,
        "do_Lowdin"                  : True,
        "is_first_time"              : {0: True},
        "rasorb_prev"                : {},
        "energies_prev"              : {},
        "energy_continuity_thresh_eV": 0.3,
    }

    res = "results_molcas"
    os.makedirs(res, exist_ok=True)

    full_id = Py2Cpp_int([0, 0])

    for i in range(50):
        print(f"\n{'='*60}")
        print(f"  Timestep {i}")
        print(f"{'='*60}")

        labels, q = cp2k.read_trajectory_xyz_file("water-md-aligned.xyz", i)
        params["timestep"] = i

        obj = molcas_compute_adi(q, params, full_id)

        obj.ham_adi.show_matrix(f"{res}/ham_adi_{i}.txt")
        obj.hvib_adi.show_matrix(f"{res}/hvib_adi_{i}.txt")
        obj.time_overlap_adi.real().show_matrix(f"{res}/st_adi_{i}.txt")
        obj.overlap_adi.real().show_matrix(f"{res}/s_adi_{i}.txt")
        print(f"  Timestep {i} → saved to {res}/")

    print("\nHappy Ending (^_^)")


if __name__ == "__main__":
    main()


[import] Loaded methods.py from: /user/someshch/project_molcas/LIBRA_MOLCAS/methods.py
Atoms  : ['O', 'H', 'H']
Natoms : 3

  Timestep 0
[DEBUG run_molcas] final fileorb = None
[DEBUG run_molcas] final thre    = 1.0e-10
[DEBUG run_molcas] final lshift  = None
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_0_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_0_traj_0.out
[run_molcas] Restart orb: None
[DEBUG] Found 4 CI vectors with 187 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, 0, -1, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,

[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 78653 (64.2%)
[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 77236 (63.0%)
  Timestep 3 → saved to results_molcas/

  Timestep 4
[DEBUG run_molcas] final fileorb = /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_3_traj_0.RasOrb
[DEBUG run_molcas] final thre    = 1.0e-10
[DEBUG run_molcas] final lshift  = None
[make_molcas_input] Using restart orbitals: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_3_traj_0.RasOrb
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_4_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_4_traj_0.out
[run_molcas] Restart orb: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_3_traj_0.RasOrb
[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State

[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,
    "min_vir": 9,
    "max_vir": 24,
    "actual_orbital_space": [
        3,
        4,
        5,
        6,
        7,
        8
    ],
    "boundaries": {
        "inactive_range": [
            1,
            2
        ],
        "active_range": [
            3,
            8
        ],
        "virtual_range": [
            9,
            24
        ]
    }
}
[ENERGY SANITY]  traj=0 step=7 All states continuous (thresh=0.3 eV)
[DEBUG] HDF5 keys in wd_molcas_itraj0/input__timestep_7_traj_0.scf.h5: ['AO_FOCKINT_MAT

[ci_overlap_general] Total det pairs : 122150 | Skipped (|CK*CL| < 1e-06): 75041 (61.4%)
[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 73525 (60.0%)
  Timestep 10 → saved to results_molcas/

  Timestep 11
[DEBUG run_molcas] final fileorb = /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_10_traj_0.RasOrb
[DEBUG run_molcas] final thre    = 1.0e-10
[DEBUG run_molcas] final lshift  = None
[make_molcas_input] Using restart orbitals: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_10_traj_0.RasOrb
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_11_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_11_traj_0.out
[run_molcas] Restart orb: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_10_traj_0.RasOrb
[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG

[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,
    "min_vir": 9,
    "max_vir": 24,
    "actual_orbital_space": [
        3,
        4,
        5,
        6,
        7,
        8
    ],
    "boundaries": {
        "inactive_range": [
            1,
            2
        ],
        "active_range": [
            3,
            8
        ],
        "virtual_range": [
            9,
            24
        ]
    }
}
[ENERGY SANITY]  traj=0 step=14 All states continuous (thresh=0.3 eV)
[DEBUG] HDF5 keys in wd_molcas_itraj0/input__timestep_14_traj_0.scf.h5: ['AO_FOCKINT_M

...

[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,
    "min_vir": 9,
    "max_vir": 24,
    "actual_orbital_space": [
        3,
        4,
        5,
        6,
        7,
        8
    ],
    "boundaries": {
        "inactive_range": [
            1,
            2
        ],
        "active_range": [
            3,
            8
        ],
        "virtual_range": [
            9,
            24
        ]
    }
}
[ENERGY SANITY]  traj=0 step=35 All states continuous (thresh=0.3 eV)
[DEBUG] HDF5 keys in wd_molcas_itraj0/input__timestep_35_traj_0.scf.h5: ['AO_FOCKINT_M

[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 79276 (64.7%)
[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 78546 (64.1%)
  Timestep 38 → saved to results_molcas/

  Timestep 39
[DEBUG run_molcas] final fileorb = /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_38_traj_0.RasOrb
[DEBUG run_molcas] final thre    = 1.0e-10
[DEBUG run_molcas] final lshift  = None
[make_molcas_input] Using restart orbitals: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_38_traj_0.RasOrb
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_39_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_39_traj_0.out
[run_molcas] Restart orb: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_38_traj_0.RasOrb
[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG

[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,
    "min_vir": 9,
    "max_vir": 24,
    "actual_orbital_space": [
        3,
        4,
        5,
        6,
        7,
        8
    ],
    "boundaries": {
        "inactive_range": [
            1,
            2
        ],
        "active_range": [
            3,
            8
        ],
        "virtual_range": [
            9,
            24
        ]
    }
}
[ENERGY SANITY]  traj=0 step=42 All states continuous (thresh=0.3 eV)
[DEBUG] HDF5 keys in wd_molcas_itraj0/input__timestep_42_traj_0.scf.h5: ['AO_FOCKINT_M

[ci_overlap_general] Total det pairs : 122150 | Skipped (|CK*CL| < 1e-06): 68776 (56.3%)
[ci_overlap_general] Total det pairs : 122500 | Skipped (|CK*CL| < 1e-06): 68589 (56.0%)
  Timestep 45 → saved to results_molcas/

  Timestep 46
[DEBUG run_molcas] final fileorb = /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_45_traj_0.RasOrb
[DEBUG run_molcas] final thre    = 1.0e-10
[DEBUG run_molcas] final lshift  = None
[make_molcas_input] Using restart orbitals: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_45_traj_0.RasOrb
[run_molcas] Running    : /projects/academic/cyberwksp21/SOFTWARE_2026/OpenMolcas/pymolcas input__timestep_46_traj_0.in
[run_molcas] Working dir: wd_molcas_itraj0
[run_molcas] Output file: wd_molcas_itraj0/input__timestep_46_traj_0.out
[run_molcas] Restart orb: /user/someshch/project_molcas/LIBRA_MOLCAS/wd_molcas_itraj0/input__timestep_45_traj_0.RasOrb
[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG

[DEBUG] Found 4 CI vectors with 350 total configurations
[DEBUG] State 1: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 2: first config = (2, 2, 1, -1, 0, 0), len = 6
[DEBUG] State 3: first config = (2, 2, 2, 0, 0, 0), len = 6
[DEBUG] State 4: first config = (2, 2, 1, -1, 0, 0), len = 6
{
    "nao": 24,
    "nmo": 24,
    "nelec": 10,
    "nocc": 2,
    "nact": 6,
    "nact_elec": 6,
    "min_occ": 1,
    "max_occ": 2,
    "min_active": 3,
    "max_active": 8,
    "min_vir": 9,
    "max_vir": 24,
    "actual_orbital_space": [
        3,
        4,
        5,
        6,
        7,
        8
    ],
    "boundaries": {
        "inactive_range": [
            1,
            2
        ],
        "active_range": [
            3,
            8
        ],
        "virtual_range": [
            9,
            24
        ]
    }
}
[ENERGY SANITY]  traj=0 step=49 All states continuous (thresh=0.3 eV)
[DEBUG] HDF5 keys in wd_molcas_itraj0/input__timestep_49_traj_0.scf.h5: ['AO_FOCKINT_M